<a href="https://colab.research.google.com/github/leahuriarte/timetransformers/blob/main/timetransformers.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Pip install required libraries

In [2]:
!pip install openmeteo-requests
!pip install requests-cache retry-requests numpy pandas
!pip install -q git+https://github.com/huggingface/transformers.git kernels accelerate peft matplotlib pyarrow scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 890.2/890.2 kB 33.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 74.2 MB/s  0:00:00
  Attempting uninstall: flatbuffers
    Found existing installation: flatbuffers 25.12.19
    Uninstalling flatbuffers-25.12.19:
      Successfully uninstalled flatbuffers-25.12.19
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8/8 [openmeteo-requests]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [requests-cache]
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


Import libraries

In [15]:
import requests
import openmeteo_requests
import pandas as pd
import requests_cache
from retry_requests import retry
import numpy as np
import torch
from datetime import date, timedelta
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

from transformers import TimesFm2_5ModelForPrediction

Get today's date, date from one week ago, date from two weeks ago

In [4]:
today = date.today()
one_week_ago = today - timedelta(days=7)
two_weeks_ago = today - timedelta(days=14)
print(today, one_week_ago, two_weeks_ago)

2026-10-27 2026-10-20 2026-10-13


In [7]:
claremont_lat = 34.13
claremont_long = -117.73

Code to get hourly weather from last week, two weeks ago

In [5]:
# Source: https://open-meteo.com/en/docs
def get_historical_weather(lat, long, start_date, end_date):
  # Setup the Open-Meteo API client with cache and retry on error
  cache_session = requests_cache.CachedSession('.cache', expire_after = -1)
  retry_session = retry(cache_session, retries = 5, backoff_factor = 0.2)
  openmeteo = openmeteo_requests.Client(session = retry_session)
  # Make sure all required weather variables are listed here
  # The order of variables in hourly or daily is important to assign them correctly below
  url = "https://archive-api.open-meteo.com/v1/archive"
  params = {
    "latitude": lat,
    "longitude": long,
    "start_date": "2026-10-10",
    "end_date": "2026-10-24",
    "hourly": "temperature_2m",
    "temperature_unit": "fahrenheit",
  }
  responses = openmeteo.weather_api(url, params = params)

  # Process  location
  response = responses[0]

  # Process hourly data.
  hourly = response.Hourly()
  hourly_temperature_2m = hourly.Variables(0).ValuesAsNumpy()

  hourly_data = {
    "date": pd.date_range(
      start = pd.to_datetime(hourly.Time(), unit = "s", utc = True),
      end =  pd.to_datetime(hourly.TimeEnd(), unit = "s", utc = True),
      freq = pd.Timedelta(seconds = hourly.Interval()),
      inclusive = "left"
    )
  }

  hourly_data["temperature_2m"] = hourly_temperature_2m

  hourly_dataframe = pd.DataFrame(data = hourly_data)
  return hourly_temperature_2m, hourly_dataframe

In [8]:
two_weeks_hourly_temp, two_weeks_hourly_dataframe = get_historical_weather(claremont_lat, claremont_long, two_weeks_ago, one_week_ago)
one_week_hourly_temp, one_week_hourly_dataframe = get_historical_weather(claremont_lat, claremont_long, one_week_ago, today)

In [9]:
#source: https://github.com/huggingface/notebooks/blob/main/examples/timesfm2_5.ipynb
device = "cpu"
model = TimesFm2_5ModelForPrediction.from_pretrained(
    "google/timesfm-2.5-200m-transformers",
    device_map=device,
)
model.eval()
print(f"Model loaded with {sum(p.numel() for p in model.parameters()):,} parameters")
print(f"Horizon length: {model.config.horizon_length}")
print(f"Context length: {model.config.context_length}")
print(f"Quantiles: {model.config.quantiles}")

config.json:   0%|          | 0.00/914 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  925MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

Model loaded with 231,289,280 parameters
Horizon length: 128
Context length: 16384
Quantiles: [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]


In [10]:
past_values = [
    torch.tensor(two_weeks_hourly_temp, dtype=torch.float32, device=device)
]

with torch.no_grad():
    outputs = model(past_values=past_values)

print(f"Mean predictions shape: {outputs.mean_predictions.shape}")
print(f"Full predictions shape: {outputs.full_predictions.shape}")

Mean predictions shape: torch.Size([1, 128])
Full predictions shape: torch.Size([1, 128, 10])


Plot forecast: one with the predicted and one with the true value

In [16]:
# source: https://github.com/huggingface/notebooks/blob/main/examples/timesfm2_5.ipynb (adapted)
def plot_forecasts(
    signals, outputs, horizon_length, true_values=None,
    quantile_idx_low=0, quantile_idx_high=-1, history_length=200
):
    """Plot history, forecasts, prediction intervals, and optional actual values."""
    n = len(signals)
    fig = make_subplots(rows=1, cols=n, subplot_titles=list(signals.keys()))

    mean_preds = outputs.mean_predictions.detach().cpu().numpy()
    full_preds = outputs.full_predictions.detach().cpu().numpy()

    palette = px.colors.qualitative.Plotly
    c_hist, c_fore, c_actual = palette[0], palette[1], palette[2]

    for idx, (name, signal) in enumerate(signals.items()):
        col = idx + 1
        show_legend = idx == 0

        context = np.asarray(signal)[-history_length:]
        forecast = mean_preds[idx, :horizon_length]
        q_low = full_preds[idx, :len(forecast), quantile_idx_low]
        q_high = full_preds[idx, :len(forecast), quantile_idx_high]

        t_hist = np.arange(len(context))
        t_pred = np.arange(len(context), len(context) + len(forecast))

        fig.add_trace(
            go.Scatter(x=t_hist, y=context, name="History",
                       line=dict(color=c_hist), legendgroup="history",
                       showlegend=show_legend),
            row=1, col=col,
        )

        # Prediction interval: invisible lower bound, upper bound filled down to it
        fig.add_trace(
            go.Scatter(x=t_pred, y=q_low, mode="lines", line=dict(width=0),
                       hoverinfo="skip", legendgroup="interval", showlegend=False),
            row=1, col=col,
        )
        fig.add_trace(
            go.Scatter(x=t_pred, y=q_high, mode="lines", line=dict(width=0),
                       fill="tonexty", fillcolor="rgba(239, 85, 59, 0.3)",
                       name="Prediction interval", hoverinfo="skip",
                       legendgroup="interval", showlegend=show_legend),
            row=1, col=col,
        )

        fig.add_trace(
            go.Scatter(x=t_pred, y=forecast, name="Forecast",
                       line=dict(color=c_fore), legendgroup="forecast",
                       showlegend=show_legend),
            row=1, col=col,
        )

        if true_values is not None and name in true_values:
            actual = np.asarray(true_values[name])[:len(forecast)]
            fig.add_trace(
                go.Scatter(x=t_pred[:len(actual)], y=actual, name="Actual",
                           line=dict(color=c_actual), legendgroup="actual",
                           showlegend=show_legend),
                row=1, col=col,
            )

        fig.add_vline(x=len(context), line=dict(dash="dash", width=1),
                      opacity=0.5, row=1, col=col)

    fig.update_layout(width=600 * n, height=400,
                      legend=dict(x=0.01, y=0.99, xanchor="left", yanchor="top"))
    return fig


fig = plot_forecasts(
    {"Temperature (°F)": two_weeks_hourly_temp},
    outputs,
    model.config.horizon_length,
    {"Temperature (°F)": one_week_hourly_temp},
)
fig.show()